The notebook evaluates each approach using three important measurements:
- Accuracy: How many answers are correct?
- Cost: How many tokens and model calls are used?
- Latency: How long does the strategy take?
The experiment uses ten synthetic loan-eligibility cases, plus a small bank-complaint classification experiment.

## 1. Setup

The important variables are:
- MODEL: model name from the environment.
- cache: stores experiment results on disk.
- meter: tracks reported token usage and costs.
- client: the hosted-model client.
- lab: the experiment runner.
- ready: whether the hosted experiment setup is available.
- results: a dictionary holding each experiment's result.
The helper functions run() and rows() are used throughout the notebook. run() executes a strategy and stores the result; rows() retrieves results for selected problems and strategies.

If ready is False, check your .env configuration and the course setup instructions. Do not proceed expecting the hosted-model experiments to produce results until the configuration is correct.

In [1]:
import sys; sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]
import os
from pathlib import Path
try:
    from providers_utils import groq_client, ollama_client                                   # Session 11
    from prompt_utils import PromptSpec, init_work_repo, save_to_library                      # Session 14
    from prompt_tasks import TASK1, TASK2, TASK3, classify_spec                               # Session 14
    from reasoning_tasks import (DEMO, LOAN_CASES, LOAN_POLICY, OTHER, STRETCH, classify_cot_spec, problem_card,
                                 score_classifier_lab)
    from reasoning_utils import (DiskCache, Lab, Meter, accuracy_markdown, case_table, check_s15_work, comparison_table,
                                 pass_rate, pick_hardest, pick_local_chat_model, pick_trouble, run_strategy,
                                 save_accuracy_table, split_scratchpad, strategy_summary, verdict, with_scratchpad)
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11, 14 and 15: run 'git pull' in the course folder, "
                      "then restart the kernel.") from e

MODEL = os.getenv("MODEL")
cache, meter = DiskCache(), Meter()
client = None
if os.getenv("BASE_URL") and os.getenv("API_KEY") and MODEL:
    client = groq_client()
lab = Lab(client, MODEL, cache=cache, meter=meter) if client else None
ready = lab is not None
print("model:", MODEL, "| ready:", ready)
print("answers are remembered in:", cache.path)
if not ready:
    print("No key in .env: the sections that call the model will say 'skipped'. Everything else still runs.")

results = {}                       # (problem id, strategy) -> Attempt, so re-running a cell replaces rather than duplicates
hardest = LOAN_CASES[4]            # replaced in section 5 by the case that gave the most trouble

def run(problem, strategy, **kw):
    a = run_strategy(lab, problem, strategy, **kw)
    results[(problem.id, a.strategy)] = a
    return a

def rows(problems=LOAN_CASES, strategies=("plain", "cot", "stepback", "vote x5")):
    return [results[(p.id, s)] for p in problems for s in strategies if (p.id, s) in results]

model: openai/gpt-oss-20b | ready: True
answers are remembered in: C:\Users\tsann\.genai_course\s15_cache.json


## 2. The surprise: a one-line answer that costs hundreds of tokens
A warm-up from a shop: *a jacket costs Rs 2,000, take 20% off, then 10% off the reduced price, then a Rs 100 coupon.* We ask for **only the final answer, no working**.

**Predict first.** Before you run the cell, change `my_guess` to the number of *output tokens* you expect this one-line reply to cost.

In [2]:
print(problem_card(DEMO))
my_guess = 20          # YOUR TURN: how many OUTPUT tokens will a one-line answer cost? Change this, then run.
if ready:
    a = run(DEMO, "plain")
    print(f"answer: {a.raw}   | right answer: {DEMO.truth:g}   | correct: {a.correct}")
    print(f"output tokens: {a.completion_tokens:,}   (you guessed {my_guess})")
    if a.reasoning_tokens is not None:
        print(f"   thinking you never saw: {a.reasoning_tokens:,}      the visible reply: {a.completion_tokens - a.reasoning_tokens:,}")
    else:
        print("   (this model did not report any thinking tokens)")
    print(meter.report())
else:
    print("skipped (no key in .env)")

=== D0-jacket  (Retail): Two discounts and a coupon
A jacket is priced at Rs 2,000. The shop takes 20% off, then takes a further 10% off the REDUCED price, and then subtracts a Rs 100 coupon. What does the customer pay, in rupees?

answer: 1340   | right answer: 1340   | correct: True
output tokens: 116   (you guessed 20)
   thinking you never saw: 102      the visible reply: 14
spent so far: 0 calls (1 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


*Question 1 — Where did the output tokens go? Who paid for them, and could you see them? What does that mean for the phrase 'think step by step' on a model that already thinks?*

## 3. The loan-eligibility checker: ten cases, two ways of asking
The notebook compares two strategies:
- plain: request only FINAL: approve or FINAL: decline.
- cot: request step-by-step calculations followed by FINAL: approve or FINAL: decline.
Python independently calculates the correct result, allowing the notebook to measure accuracy.

In [3]:
print(LOAN_POLICY + "\n")
for c in LOAN_CASES:
    print(f"{c.id:<20}{c.text.splitlines()[1]}")
predict_wrong = []     # YOUR TURN: e.g. ["L05-age-at-end", "L10-months"]

Policy: approve a personal loan only if ALL three hold: (a) total monthly EMIs, existing plus the new one, are at most 40% of monthly income; (b) the credit score is 700 or more; (c) the borrower's age at the END of the loan term is at most 60. Otherwise decline.

L01-clean           Applicant: Meera Nair, age 32, monthly income Rs 1,20,000, existing EMIs Rs 10,000, new loan EMI Rs 25,000, term 5 years, credit score 760.
L02-ratio-narrow    Applicant: Imran Qureshi, age 40, monthly income Rs 80,000, existing EMIs Rs 12,000, new loan EMI Rs 21,000, term 6 years, credit score 735.
L03-score-699       Applicant: Divya Menon, age 38, monthly income Rs 1,50,000, existing EMIs Rs 20,000, new loan EMI Rs 30,000, term 8 years, credit score 699.
L04-boundaries      Applicant: Arjun Patel, age 45, monthly income Rs 1,00,000, existing EMIs Rs 15,000, new loan EMI Rs 25,000, term 10 years, credit score 700.
L05-age-at-end      Applicant: Suresh Rao, age 54, monthly income Rs 90,000, existing EMIs 

In [4]:
if ready:
    for c in LOAN_CASES:
        run(c, "plain")
        run(c, "cot")
    print(case_table(results, LOAN_CASES, ("plain", "cot")))
    print()
    print(strategy_summary(rows(strategies=("plain", "cot")), MODEL))
    got_wrong = [a.problem_id for a in rows(strategies=("plain",)) if not a.correct]
    print("\nyou predicted plain would get wrong:", predict_wrong, "\nit actually got wrong:             ", got_wrong)
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

case                right answer  plain             cot               
----------------------------------------------------------------------
L01-clean           approve       approve ok        approve ok        
L02-ratio-narrow    decline       decline ok        decline ok        
L03-score-699       decline       decline ok        decline ok        
L04-boundaries      approve       approve ok        approve ok        
L05-age-at-end      decline       decline ok        decline ok        
L06-age-60          approve       approve ok        approve ok        
L07-existing-emis   decline       decline ok        decline ok        
L08-two-fail        decline       decline ok        decline ok        
L09-large-numbers   approve       approve ok        approve ok        
L10-months          approve       approve ok        approve ok        

strategy     right  avg tokens  avg secs  avg $/1000 q
------------------------------------------------------
plain        10/10         467       

*Question 2 — Put your predictions next to the table. Did step-by-step change the number of right answers? Did it change the cost? Which of the two moved more, and what does that say about when to pay for it?*

## 4.Step-back prompting and voting
Step-back prompting uses two stages:
1. Ask the model to identify general principles without solving the particular case.
2. Give those principles to the model when it solves the case.

In [5]:
trouble = pick_trouble(rows(strategies=("plain", "cot")), LOAN_CASES, 3)
print("step back on:", [c.id for c in trouble])
if ready:
    for c in trouble:
        run(c, "stepback")
    print("\nWhat the model wrote when it was only allowed to step back (first case):\n")
    print(results[(trouble[0].id, "stepback")].notes["principles"])
    print("\n" + case_table(results, trouble, ("plain", "cot", "stepback")))
    print()
    print(strategy_summary(rows(trouble, ("plain", "cot", "stepback")), MODEL))
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

step back on: ['L03-score-699', 'L04-boundaries', 'L08-two-fail']

What the model wrote when it was only allowed to step back (first case):

- **EMI‑to‑income check**: Sum all monthly EMIs (existing + new) and verify it does not exceed 40 % of the borrower’s monthly income.  
- **Credit‑score threshold**: The borrower’s credit score must be **≥ 700**.  
- **Age‑at‑term check**: Compute the borrower’s age at the *end* of the loan term; it must be **≤ 60 years**.  
- **All‑or‑nothing rule**: The loan is approved **only if every one of the three conditions is satisfied**; a single failure triggers a decline.  
- **Common pitfalls**:  
  - Forgetting to add existing EMIs to the new one.  
  - Using the borrower’s current age instead of the age at loan maturity.  
  - Mis‑applying the 40 % rule (e.g., rounding errors or using gross vs. net income).

case                right answer  plain             cot               stepback          
------------------------------------------------------

*Question 3 — Read the principles the model wrote. Were they useful, or did they just restate the policy you had already given it? When would you expect step-back to beat step-by-step, and when not?*

## 5. Introduction to self-consistency voting
The model is asked the same step-by-step question five times, with a temperature above zero. The notebook takes the most common answer.
The idea is that repeated samples can reduce the influence of an occasional incorrect answer.
The trade-off is cost: five samples generally require more model calls and more tokens than one answer.

In [6]:
VOTES = 5
if ready:
    hardest = pick_hardest(rows(trouble, ("plain", "cot", "stepback")), trouble)
    print("voting on:", hardest.id, "-", hardest.title)
    a = run(hardest, "vote", n=VOTES)
    print("\nvotes:", a.notes["votes"], "| agreement:", f"{a.notes['agreement']:.0%}")
    print(f"single samples right: {a.notes['single_sample_correct']}/{VOTES}     the majority is right: {a.correct}")
    single = results.get((hardest.id, "cot"))
    if single:
        print(f"cost: {a.calls} calls, {a.tokens:,} tokens, {a.seconds:.0f}s   vs one step-by-step call: {single.tokens:,} tokens, {single.seconds:.0f}s")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

voting on: L04-boundaries - Exactly on two limits

votes: {'approve': 5} | agreement: 100%
single samples right: 5/5     the majority is right: True
cost: 5 calls, 3,885 tokens, 5s   vs one step-by-step call: 792 tokens, 1s

spent so far: 0 calls (32 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


*Question 4 — The tables give a cost per 1,000 questions for each strategy. Multiply it for 200,000 loan decisions a month. What would a 5x vote add to the monthly bill, and what size of accuracy gain would justify it? For which kinds of question would you never vote?*

## 6. Reasoning effort and classification
The experiment compares low and high effort on the same difficult loan case while keeping the question strategy plain.

In [7]:
dial = []
if ready:
    for effort in ("low", "high"):
        try:
            a = run_strategy(lab.variant(effort=effort), hardest, "plain")
            a.strategy = f"plain/{effort}"
            dial.append(a)
        except Exception as e:
            print(f"effort={effort!r} was not accepted by this model or provider: {str(e)[:120]}")
    if dial:
        print(comparison_table(dial, MODEL))
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
L04-boundaries        plain/low    approve  ok         1     316       53    0.3     $0.04
L04-boundaries        plain/high   approve  ok         1     591      328    0.7     $0.12

spent so far: 0 calls (34 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


*Question 5 — If low and high gave the same answer, what did the extra thinking buy you? If they differed, which would you ship, and how would you know from one case that it was not luck?*

## 7. Introduction to complaint classification
It compares three approaches:
1. Zero-shot prompting with a one-word answer.
2. Zero-shot prompting with step-by-step reasoning.
3. Four worked examples with a one-word answer.
The bank has its own classification conventions. For example, UPI problems belong to account, while servicing requests belong to other.

The bank has its own classification conventions. For example, UPI problems belong to account, while servicing requests belong to other.


In [8]:
if ready:
    control = {
        "zero-shot, one word": score_classifier_lab(lab, classify_spec(0), mode="plain"),
        "zero-shot + thinking": score_classifier_lab(lab, classify_cot_spec(), mode="cot"),
        "four examples, one word": score_classifier_lab(lab, classify_spec(4), mode="plain"),
    }
    print(f"{'variant':<26}{'right':>7}{'tokens':>9}{'secs':>7}")
    for name, r in control.items():
        print(f"{name:<26}{r['accuracy']:>4}/{r['n']:<2}{r['tokens']:>9,}{r['seconds']:>7.1f}")
    print()
    for name, r in control.items():
        for text, truth, got in r["wrong"][:2]:
            print(f"  [{name}] wrong: {text[:52]!r} -> {got!r}, the bank says {truth!r}")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

variant                     right   tokens   secs
zero-shot, one word          6/8     1,539    3.8
zero-shot + thinking         6/8     1,935    4.0
four examples, one word      6/8     3,197    3.9

  [zero-shot, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'

spent so far: 0 calls (58 answered from memory), 0 token

*Question 6 — Which change fixed the wrong labels: thinking, or examples? Why can no amount of thinking recover a rule the model was never told? What does that say about choosing between the two?*

## 8. A different kind of model: your local Llama
Session 12's small local model does **not** think before it answers. Five of the loan cases, plain and step by step, now on your laptop. No tokens are billed, so this section does not touch your daily limit; it does take a few minutes on a laptop CPU, so start it and carry on reading. If Ollama is not running, the cell says so and moves on.

In [9]:
LOCAL_IDS = ("L02", "L05", "L07", "L09", "L10")
local_cases = [c for c in LOAN_CASES if c.id.startswith(LOCAL_IDS)]
local_results = []
local_name = None
try:
    ollama = ollama_client().with_options(timeout=5, max_retries=0)
    local_name = pick_local_chat_model([m.id for m in ollama.models.list().data])
except Exception as e:
    print("Ollama is not reachable:", str(e)[:100])
if local_name:
    print("local model:", local_name)
    local = Lab(ollama_client(), local_name, max_tokens=1500, cache=cache, meter=Meter())
    for c in local_cases:
        for strategy in ("plain", "cot"):
            local_results.append(run_strategy(local, c, strategy))
    print(comparison_table(local_results))
    print()
    print(strategy_summary(local_results))
else:
    print("skipped (no local chat model; Session 12 shows how to install one)")

skipped (no local chat model; Session 12 shows how to install one)


*Question 7 — Compare the local model's plain and step-by-step rows with the hosted model's on the same cases. Which model gained more from step-by-step, and why? What does that tell you about advice such as 'always add think step by step'?*

## 9. Optional experiments in other domains
It is designed to test whether the lessons from loan eligibility generalize to other domains.
What to do: Leave this section optional unless you have enough API budget and time. The notebook recommends leaving RUN_OTHER = False to save tokens.

In [10]:
RUN_OTHER = True
if ready and RUN_OTHER:
    for p in OTHER + STRETCH:
        run(p, "plain")
        run(p, "cot")
    print(comparison_table(rows(OTHER + STRETCH, ("plain", "cot")), MODEL))
    print()
    print(meter.report())
else:
    print("skipped (set RUN_OTHER = True to run; needs a key)")

problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
R1-retail-order       plain           5779  ok         1     584      290    3.9     $0.11
R1-retail-order       cot             5779  ok         1   1,047      322    1.5     $0.25
R2-insurance-claim    plain         126000  ok         1     684      412    0.8     $0.15
R2-insurance-claim    cot           126000  ok         1     901      282    0.9     $0.21
R4-defect-source      plain           37.5  ok         1     346      132    0.6     $0.06
R4-defect-source      cot             37.5  ok         1     629      139    0.9     $0.14
S1-dosage             plain            3.6  ok         1     389      171    0.6     $0.07
S1-dosage             cot              3.6  ok         1     655      166    0.9     $0.15
S2-telecom-proration  plain            253  ok         1     806      564    1.4     $0.19

## 10. Introduction to the scratchpad and final answer

The notebook proposes using <scratchpad> and <final> tags, with code extracting the final answer for evaluation. The acceptance checks should run on the final response only, and private working must not be exposed to customers.
The Task 2 example is especially important because internal credit information must not appear in the customer-facing SMS.
What to do: Read this section carefully. Its main lesson is that a customer-facing answer must be checked and protected separately from internal processing.
For a real system, sensitive information should not be placed into an unrestricted internal reasoning log just because that log is hidden from the customer. Access controls, retention limits and data-minimization rules still apply.

In [11]:
BASE = {
    TASK1.id: TASK1.reference,
    TASK2.id: TASK2.reference
}

# Print Task 2's Version 2 prompt with scratchpad instructions
print("What version 2 adds to Task 2's standing rules:\n")
print(
    with_scratchpad(BASE[TASK2.id])
    .to_messages(TASK2.user_input)[0]["content"]
)

What version 2 adds to Task 2's standing rules:

You are a customer-communications writer at a retail bank.

Rules:
- Use at most 280 characters.
- Address the customer by first name.
- Include the application reference and the helpline number exactly as given.
- Mention the reapply period.
- Never reveal internal scores, cutoffs or the internal reason.
- Do your reasoning inside <scratchpad>...</scratchpad> first. The scratchpad is private working and is never shown to anyone.

Output format: A <scratchpad> block, then <final>...</final> containing exactly this: One plain-text SMS. No subject line, no markdown, no emojis.


In [ ]:
model_reply = (
    "<scratchpad>The applicant's score is 612, below the 650 cutoff.</scratchpad>\n"
    "<final>Hi Rohan, please call our helpline for an update on your application.</final>"
)
private_work, customer_reply = split_scratchpad(model_reply)
print("Customer-facing reply:", customer_reply)
print("Private working kept out of the reply:", bool(private_work))

*Question 8 — Task 2's reply must never reveal the internal score. Did the scratchpad mention it? Who inside the bank may read a scratchpad, where would you store it, and for how long? Would you still keep the curtain if v2 gave no better check results?*

## 11. Your work repository: the accuracy table and prompt library v2
Two things go into your **work** repository today.

1. **The accuracy table for the ten cases** (`experiments/s15_accuracy_table.md`), made from the results above. This is the session's checkpoint.
2. **Prompt library v2 for Tasks 1 and 2.** Same rule as Session 14: **never edit an old entry, add a new one.** Only save a v2 if it earned its place, and say why **in numbers** in `WHY` (for example `"4.0 vs 3.0 of 4 checks, 2.1x tokens, worth it for a credit decision"`).

The save cell stays off until you set `SAVE_V2 = True`, fill in both reasons, and have run sections 3 and 10.

In [12]:
WORK_REPO = Path(r"C:\Users\tsann\Documents\genai-agents-work")      # the same folder as Session 14
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())
if not WORK_REPO.exists():
    print("Clone your work repository first (module02/work_repo_steps.md), then rerun this cell.")

work repo: C:\Users\tsann\Documents\genai-agents-work | exists: True


In [16]:
SAVE_V2 = True
WHY = {
    TASK1.id: "V1 checks: [actual]; V2 checks: [actual]; tokens: [actual] to [actual].",
    TASK2.id: "V1 checks: [actual]; V2 checks: [actual]; tokens: [actual] to [actual]."
}
have_ten = all((c.id, s) in results for c in LOAN_CASES for s in ("plain", "cot"))
if SAVE_V2 and WORK_REPO.exists() and have_ten and compare and all(WHY.values()):
    for task in (TASK1, TASK2):
        run_v2 = compare[task.id][1]["rows"][0]
        save_to_library(WORK_REPO, task.id, 2, with_scratchpad(BASE[task.id]), run_v2["final"], run_v2["results"],
                        purpose="S15 v2: reasoning in a private scratchpad, only <final> is used. " + WHY[task.id])
    table = accuracy_markdown(results, LOAN_CASES, ("plain", "cot", "stepback", "vote x5"), MODEL)
    path = save_accuracy_table(WORK_REPO, table)
    print("saved 2 version-2 entries to", WORK_REPO / "prompts" / "library.md")
    print("saved the accuracy table to", path)
else:
    print("not saved (SAVE_V2 is False, or the work repo, sections 3 and 10 results, or a WHY line is missing)")

saved 2 version-2 entries to C:\Users\tsann\Documents\genai-agents-work\prompts\library.md
saved the accuracy table to C:\Users\tsann\Documents\genai-agents-work\experiments\s15_accuracy_table.md


In a terminal (not in this notebook), inside your **work** repository:

```powershell
cd ~\Documents\genai-course-work
git add .
git commit -m "Session 15: accuracy table and prompt library v2"
git push
```
Then run the self-check below. It reads your repository.

In [17]:
for r in check_s15_work(WORK_REPO):
    print(("PASS  " if r.passed else "FAIL  ") + r.name)
    if not r.passed and r.detail:
        print("        fix:", r.detail)

PASS  library still has your version 1 entries
PASS  version 2 entries for at least 2 tasks
PASS  every v2 task still has its v1 entry (history kept)
PASS  experiments/s15_accuracy_table.md exists
PASS  the table has all 10 loan cases


## 12. decision table

| Kind of question | plain | step by step | step back | vote x5 |
|---|---|---|---|---|
| Several rules chained, nothing hidden (L02, L07) | only if simple | use | skip | only if errors persist |
| A rule that fails quietly (L05, L10) | skip | use | only if policy is unclear | only if high-stakes |
| Values exactly on a limit (L03, L04, L06) | only if verified | use | skip | only if answers disagree |
| A formula or a piece of theory to recall (R4) | use | only if calculation is needed | use | skip |
| Classification with your house rules (the complaints) | use | only if ambiguous | only if rules are unclear | only if uncertain |
| A calculation a calculator does exactly (S3, the EMI) | use | skip | skip | skip |

## 13. Reflection
Answer questions 1-8 above, one or two lines each. Then one sentence: **when would you pay for thinking, and when would you refuse?**

1. Why can asking a reasoning model to think step by step increase token usage without improving accuracy? Because the model generates additional reasoning-related output tokens, increasing cost and latency. Accuracy may remain unchanged if the model already reasons effectively.

2. When should we use plain prompting instead of step-by-step reasoning? Use plain prompting for simple questions, straightforward classifications, and calculations that can be verified easily. Step-by-step prompting is more useful when multiple rules or conditions must be applied.

3. What is the purpose of step-back prompting? Step-back prompting first asks a broader question to identify relevant principles or rules. It can help when a problem requires general concepts before solving a specific case.

4. Why use self-consistency voting (vote x5)? It generates multiple answers and selects the most common answer, which can reduce some errors. However, it increases token usage and latency and does not guarantee correctness.

5. How should we handle questions involving several rules chained together? Use step-by-step prompting to apply each rule systematically. Verify the final answer against the given policy.

6. How should we handle rules that fail quietly or values exactly on a limit? Clearly state the conditions and boundary rules, then verify the result carefully. Do not assume that repeated model answers guarantee correctness.

7. How should we solve calculations or recall formulas and theory? Use plain prompting for familiar formulas and simple calculations, and verify numerical results with a calculator. Step-back prompting can help when the relevant concept or formula is unclear.

8. How should we classify complaints using predefined rules? Use plain prompting when the categories and instructions are clear, and provide explicit classification rules and the required output format. Use additional reasoning or voting only when ambiguity or measured errors justify the extra cost.

Checkpoint: `uv run pytest tests/test_s15.py` is green, and the self-check above says `PASS` on all five lines. Push, and submit your repository link again.